In [1]:
import torch
from türkçe_dil_modelim.tokenizer import Tokenizer
from türkçe_dil_modelim.model import Model

tokenizer= Tokenizer("türkçe_dil_modelim/tokenizer.json")
prompt="Benim adım Burak"
prompts= [
    "Benim adım Burak",
    "senin adın ",
    "nerede yaşıyorsun"
]# eğer batch batch eğitmek istiyorsam bu şekilde vermem gerekiyo verileri.
#mesela resim verisinde bu durum her bir resim özelinde 2D bir liste. cümle için ise 1d bir liste oluyor
tokens = tokenizer.encode(prompts[0])
tokens.to("cpu")


tensor([1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
        1153])

In [2]:
batch_tokens= tokenizer.encode_batch(prompts,32)

In [3]:
batch_tokens.shape

torch.Size([3, 32])

In [4]:
batch_tokens

tensor([[1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
         1153,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 497,  622,  373,  342, 1148,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 534, 1162,  462,  622,  371,  297,  425,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627]])

In [8]:
context_length= 32
torch.manual_seed(1)

model= Model(vocab_size=tokenizer.get_vocab_size(),embedding_dim=12,num_heads=4,context_length=context_length,num_layers=8,device="cpu")

In [9]:
out= model(batch_tokens)
out=out.squeeze(0)
out.shape

torch.Size([3, 32, 2881])

In [10]:
out=model.generate(tokens,2, temperature=10,top_k=12,top_p=0.4)
tokenizer.decode(out)

'Benim adım Burakgeriı'

In [11]:
token_ids= tokenizer.encode(text)
len(token_ids), type(token_ids)

(693, torch.Tensor)

In [12]:
from türkçe_dil_modelim.text_dataset import create_data_loader

stride=12

train_data_loader= create_data_loader(token_ids=token_ids.tolist(),context_length=context_length,stride=stride,batch_size=3,shuffle=True)

In [13]:
parameters_count= sum(p.numel() for p in model.parameters())
print(parameters_count)

print(model)

55712
Model(
  (embedding): Embedding(
    (embedding): Embedding(1376, 12)
  )
  (layers): Sequential(
    (0): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (layer_norm1): LayerNorm()
      (layer_norm2): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=48, bias=True)
        (up_proj): Linear(in_features=12, out_features=48, bias=True)
        (down_proj): Linear(in_features=48, out_features=12, bias=True)
        (gelu): GELU()
      )
    )
    (1): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_feat

In [14]:
out= model(batch_tokens)
out,out.flatten(),out.flatten(0,1),out.flatten(0,1).shape

(tensor([[[ 0.2742,  0.5289, -0.0548,  ..., -0.0746, -0.6976,  0.2787],
          [ 0.0202, -0.4472,  0.9803,  ..., -0.8815,  0.1148, -1.4019],
          [-0.2123, -0.1348,  0.3730,  ..., -0.6394, -0.9094, -0.5363],
          ...,
          [-0.0138,  0.5016,  0.3422,  ..., -0.6926,  0.2629,  0.8766],
          [-0.2290,  0.2922,  0.2491,  ..., -0.4324,  0.5385,  0.9477],
          [-0.1798,  0.0876,  0.2358,  ..., -0.2111,  0.7711,  0.7074]],
 
         [[ 0.2178,  0.1661, -0.2212,  ..., -0.0480, -0.2023, -0.2502],
          [ 0.2254,  0.2809,  0.3346,  ...,  0.0655,  1.3672,  0.4926],
          [-0.5262,  0.3214,  0.1533,  ..., -0.6147,  0.4229,  1.1262],
          ...,
          [-0.0138,  0.5016,  0.3422,  ..., -0.6926,  0.2629,  0.8766],
          [-0.2290,  0.2922,  0.2491,  ..., -0.4324,  0.5385,  0.9477],
          [-0.1798,  0.0876,  0.2358,  ..., -0.2111,  0.7711,  0.7074]],
 
         [[-0.4560, -0.6097,  0.4711,  ..., -0.6506,  1.2569, -0.4507],
          [-0.8877, -0.1470,

In [15]:
for i , (X,Y) in enumerate(train_data_loader):
    print(X.shape,Y.shape,Y.flatten().shape)
    break

torch.Size([3, 32]) torch.Size([3, 32]) torch.Size([96])


In [ ]:
import torch.nn as nn 
device="cpu"
loss_fn= nn.CrossEntropyLoss()

optimizer= torch.optim.AdamW(model.parameters(),lr=1e-3)

epochs= 20

for epoch in range(epochs):
    total_loss=0
    for i, (X,Y) in enumerate(train_data_loader):
        X= X.to(device)
        Y= Y.to(device)

        pred= model(X)
        loss= loss_fn(pred.flatten(0,1),Y.flatten())
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    average_loss= total_loss/ len(train_data_loader)
    print(f"Epoch {epoch+1} loss: {loss.item()} average_loss: {average_loss}")

In [ ]:
#temperature , top_k , top_p


In [ ]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor, dim=-1)

_, max_index = torch.max(probs, dim=-1)
next_token = max_index.item()

In [ ]:
probs.squeeze(0) # her zaman en yüksek olasılıklıyı seçmesin diye temperature, top_k, top_p gibi parametreleri de kullanıcaz

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
next_token

0

In [ ]:
probs= probs.squeeze(0)

In [ ]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
import torch

# 3 token için olasılık dağılımı
probs = torch.tensor([0.1, 0.3, 0.6])

# 1 token seç
sample = torch.multinomial(probs, 1)


counts= [0,0,0]
for i in range(100000): # bu kısımda deneme sayısı arttıkça gerçek olasılıklarıyla aynı değer olmaya yakınlaşacak çıktı
    sample= torch.multinomial(probs,1).item()
    counts[sample]+=1
print(counts)

[9964, 30021, 60015]


[[[1179,
   372,
   1148,
   353,
   622,
   373,
   342,
   1152,
   622,
   1179,
   323,
   373,
   1153,
   179,
   305]]]

In [ ]:
out_tensor

tensor([[1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
          323.,  373., 1153.,  179.,  305.]])

In [ ]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor/1.5, dim=-1) # temperature denemesi

probs= probs.squeeze(0)

sample_count={}
for _ in range(10000):
    sample= torch.multinomial(probs,1)
    sample_count[sample.item()]= sample_count.get(sample.item(),0)+1
sample_count

{9: 4948, 0: 5052}

In [ ]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
temperature=0.1 # temperature değeri 1'den büyük olduğunda olasılıklar birbirine yaklaştığından modelin farklı eyler deme olasılığı artar
#1'den küçükse model daha determenistik olur
out_tensor = torch.tensor([out], dtype=torch.float32)

adjusted_out= out_tensor/temperature
torch.softmax(adjusted_out,dim=-1)

tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [ ]:
outs= {}

for _ in range (100):
    response= model.generate(tokens,max_new_tokens=2,temperature=0.4)
    decode= tokenizer.decode(response)
    outs[decode]= outs.get(decode,0) +1

outs


{'Benim adım Burakbotanikakısa': 1,
 'Benim adım Burakaşçıfibroyad': 1,
 'Benim adım Burakhikingçarşamba': 1,
 'Benim adım Burakkırıküyorlar': 1,
 'Benim adım BurakfiltirlemekTennis': 1,
 'Benim adım Buraküyorumcuma': 1,
 'Benim adım Burakdamatavukat': 1,
 'Benim adım BurakAmazonparlak': 1,
 'Benim adım Buraktemizlemekooforit': 1,
 'Benim adım Burakcı': 1,
 'Benim adım Burakflüoresanlaşmaküyle': 1,
 'Benim adım Burakbahçeiğne': 1,
 'Benim adım Burakkapıçözelmek': 1,
 'Benim adım Buraküyorumsarcoma': 1,
 'Benim adım Burakmicroservicesgüzel': 1,
 'Benim adım Burak.var': 1,
 'Benim adım Buraketgöstermek': 1,
 'Benim adım BurakamcaHybrid': 1,
 'Benim adım Burakuyorsunuzmi': 1,
 'Benim adım Burakşarkıaids': 1,
 'Benim adım Burakneredenkoyu': 1,
 'Benim adım BurakyaniParkinson': 1,
 'Benim adım BurakkalkamkAmerica': 1,
 'Benim adım Burakyanmaokul': 1,
 'Benim adım Buraksmsh': 1,
 'Benim adım Burakheykeltıraşlıkkilit': 1,
 'Benim adım Burakhikinghayvancılık': 1,
 'Benim adım Burakaşçısebze': 

In [ ]:
out_tensor

tensor([1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
         323.,  373., 1153.,  311.,   59.])

In [ ]:
sorted_outs= sorted(out_tensor.tolist(), reverse=True)
sorted_indexes=[]
top_k=10
temperature= 0.5
for so in sorted_outs[:top_k]:
    so_index= out_tensor.tolist().index(so)
    sorted_indexes.append(so_index)
sorted_outs= torch.tensor(sorted_outs[:top_k])
adjusted_out= torch.softmax(sorted_outs/temperature,dim=-1)
sorted_outs, sorted_indexes, adjusted_out

(tensor([1179., 1179., 1153., 1152., 1148.,  622.,  622.,  373.,  373.,  372.]),
 [0, 0, 12, 7, 2, 4, 4, 5, 5, 1],
 tensor([5.0000e-01, 5.0000e-01, 1.3051e-23, 1.7663e-24, 5.9253e-28, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]))

In [ ]:
def top_p(self,logits,top_p):
    sorted_logits, sorted_indices= torch.sort(logits,descending=-1)
    cumulative_probs= torch.cumsum(torch.softmax(sorted_logits,dim=-1),dim=-1)
    r= cumulative_probs>top_p
    r[...,1:]=r[...,:-1].clone()
    r[...,0]=False

    sorted_logits[r]=-float('inf') 
    filtered_logits= torch.full_like(logits,-float('inf'))

    filtered_logits.scatter_(0,sorted_indices,sorted_logits)
    return filtered_logits



In [ ]:
import torch
a= torch.tensor([[1.0,2.0,3.0],[1.0,5.0,3.0],[7.0,2.0,3.0]]) #logit
#topk

temp=1
a/=temp

b=torch.softmax(a,dim=-1)#top

b


tensor([[0.0900, 0.2447, 0.6652],
        [0.0159, 0.8668, 0.1173],
        [0.9756, 0.0066, 0.0179]])

In [ ]:
import torch
from türkçe_dil_modelim.tokenizer import Tokenizer

# Tokenizer'ı yükle
tokenizer = Tokenizer(
    vocab_file="/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json",
    auto_learn=True
)

print(f"Başlangıç vocab boyutu: {tokenizer.get_vocab_size()}")


wiki_file = "/workspaces/llm-from-scratch/data/wikipedia_tr_5m_words.txt"
batch_size = 1000  
context_length = 1000  


def process_wikipedia(file_path, tokenizer, batch_size=1000, context_length=50):
    total_lines = 0
    updated_vocab = tokenizer.get_vocab_size()
    
    with open(file_path, "r", encoding="utf-8") as f:
        batch_texts = []
        for line in f:
            line = line.strip()
            if line:  # boş satırları atla
                batch_texts.append(line)
                total_lines += 1

            if len(batch_texts) == batch_size:
                # batch encode
                encoded = tokenizer.encode_batch(batch_texts, context_length)
                # opsiyonel: batch decode ile kontrol
                # decoded = [tokenizer.decode(seq) for seq in encoded]
                batch_texts = []

                # her batch sonunda vocab büyüklüğünü yazdır
                new_vocab = tokenizer.get_vocab_size()
                if new_vocab != updated_vocab:
                    print(f"{total_lines} satır işlendi, vocab büyüklüğü: {new_vocab}")
                    updated_vocab = new_vocab

        # Son batchi işle
        if batch_texts:
            encoded = tokenizer.encode_batch(batch_texts, context_length)
            new_vocab = tokenizer.get_vocab_size()
            if new_vocab != updated_vocab:
                print(f"{total_lines} satır işlendi, vocab büyüklüğü: {new_vocab}")

    # İşlem bitince vocab'ı kaydet
    tokenizer.save_vocab()
    print(f"Tüm veri işlendi. Final vocab boyutu: {tokenizer.get_vocab_size()}")


process_wikipedia(wiki_file, tokenizer, batch_size=batch_size, context_length=context_length)


Başlangıç vocab boyutu: 1383
1000 satır işlendi, vocab büyüklüğü: 2004
2000 satır işlendi, vocab büyüklüğü: 2476
3000 satır işlendi, vocab büyüklüğü: 2801
3384 satır işlendi, vocab büyüklüğü: 2882
Tüm veri işlendi. Final vocab boyutu: 2882


In [1]:
import torch
from türkçe_dil_modelim.tokenizer import Tokenizer
from türkçe_dil_modelim.text_dataset import TextDataset
from torch.utils.data import DataLoader
from pathlib import Path



tokenizer = Tokenizer("türkçe_dil_modelim/tokenizer.json")
wiki_file = "/workspaces/llm-from-scratch/data/wikipedia_tr_5m_words.txt"

batch_files = TextDataset.process_and_save_batches(
    file_path=wiki_file,
    tokenizer=tokenizer,
    context_length=32,
    stride=1,
    batch_size=64,
    save_dir="wikipedia_batches"
)

for batch_file in batch_files[:2]:  # örnek olarak ilk 2 batch
    batch_data = torch.load(batch_file)
    x, y = batch_data["input"], batch_data["target"]
    print("Input shape:", x.shape)
    print("Target shape:", y.shape)


: 

In [1]:
import re

input_file = "/workspaces/llm-from-scratch/data/wikipedia_tr_5m_words.txt"
output_file = "/workspaces/llm-from-scratch/data/wikipedia_formatted.txt"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Küçük harfe çevir (istersen kaldırabilirsin)
text = text.lower()

# Fazla boşlukları temizle
text = re.sub(r"\s+", " ", text)

# Cümlelere böl (., !, ?)
sentences = re.split(r'(?<=[.!?])\s+', text)

clean_sentences = []

for sentence in sentences:
    sentence = sentence.strip()

    # Çok kısa ve anlamsız parçaları at
    if len(sentence) < 10:
        continue

    # Cümle sonundaki nokta vs kalsın
    clean_sentences.append(f"<başla> {sentence} <bitiş>")

with open(output_file, "w", encoding="utf-8") as f:
    for s in clean_sentences:
        f.write(s + "\n")

print("Formatlama tamamlandı.")
print("Toplam cümle:", len(clean_sentences))


Formatlama tamamlandı.
Toplam cümle: 326623


In [4]:
from türkçe_dil_modelim.tokenizer import Tokenizer

tokenizer = Tokenizer("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")

input_file = "/workspaces/llm-from-scratch/data/wikipedia_formatted.txt"
output_file = "/workspaces/llm-from-scratch/data/wikipedia_encoded.txt"

with open(input_file,"r",encoding="utf-8") as f:
    lines= f.readlines()

with open(output_file,"w",encoding="utf-8") as out:
   
    for line in lines:
        token_ids=tokenizer.encode(line)
        token_ids_str = " ".join(map(str, token_ids.tolist()))
        out.write(token_ids_str + "\n")

tokenizer.save_vocab()

print("Encode işlemi tamamlandı.")


Encode işlemi tamamlandı.
